# 4 — XFOIL Profile Simulation

**Author:** Héctor Fernández Pinacho  
**Project:** IDEAL Lab — Bachelor Thesis Propeller Design Pipeline

---

## Purpose

This notebook performs the aerodynamic profile simulation stage of the propeller pipeline.

It reads the NACA profile codes generated in the previous pipeline stage and runs XFOIL for each required airfoil/Reynolds-number pair. The resulting polar files are saved in:

```text
./xfoil_polars
```

The processed aerodynamic coefficient table is saved as:

```text
./csv/xfoil_profile_simulation.csv
```

This notebook only handles profile-level aerodynamic simulation. It does not generate the LHS geometry, estimate mass, generate CAD files, or run QPROP.


## 1. Method overview

Each propeller configuration contains several NACA 4-digit profiles. The relevant profile set depends on how the aerodynamic blade is defined:

- **Hub-root mode** uses `hub`, `mid`, and `outer` profiles.
- **Inner-root mode** uses `inner`, `mid`, and `outer` profiles.

The hub-root mode is the recommended setting for QPROP when the inner profile lies inside the solid hub. In that case, the hidden inner profile is only a CAD loft-control section, while the aerodynamic blade should start at the hub outer radius.

For each selected station, the local Reynolds number is computed from the local tangential velocity and chord:

$$
Re = \frac{\omega r c}{\nu}
$$

with:

$$
\omega = \frac{2\pi \cdot RPM}{60}
$$

The Reynolds number is clipped to a selected range and rounded to a fixed interval. This reduces the number of unique XFOIL runs and makes polar caching effective.

For each unique pair:

$$
(\text{NACA code}, Re)
$$

XFOIL is run once. If the same pair appears again, the cached polar file is reused.

The analysis uses three robustness levels:

1. **Viscous XFOIL polar** — preferred result.
2. **Inviscid XFOIL polar** — used if the viscous polar fails or cannot be parsed reliably.
3. **Analytical fallback** — used only if XFOIL cannot provide a usable result.


In [1]:
# Imports

from __future__ import annotations

import os
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import groupby
from operator import itemgetter
from pathlib import Path
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd

try:
    from scipy.interpolate import CubicSpline
except ImportError as exc:
    raise ImportError(
        "This notebook requires scipy for the cubic spline interpolation used "
        "to evaluate the hub chord when hub-root mode is active."
    ) from exc

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []


c:\Users\hecto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Constants and file paths

All parameters that control this notebook are defined here.

The most important switch is:

```python
USE_HUB_PROFILE_FOR_QPROP = True
```

When it is `True`, the notebook runs the XFOIL batch for:

```text
hub, mid, outer
```

When it is `False`, the notebook runs the XFOIL batch for:

```text
inner, mid, outer
```

This makes the notebook compatible with both interpretations of the blade root:

- **CAD construction view:** the inner profile is part of the loft definition.
- **Aerodynamic simulation view:** the blade starts at the hub outer radius.

For QPROP, the hub-root option is usually the cleaner choice when the inner profile is inside the hub.


In [2]:
# -----------------------------
# Input/output paths
# -----------------------------

BASE_DIR = Path.cwd()
CSV_DIR = BASE_DIR / "csv"

GEOMETRY_CSV_PATH = CSV_DIR / "prop_geometrical_params.csv"
NACA_CODES_CSV_PATH = CSV_DIR / "naca_codes.csv"
OUTPUT_CSV_PATH = CSV_DIR / "xfoil_profile_simulation.csv"

POLAR_DIR = BASE_DIR / "xfoil_polars"
XFOIL_PATH = BASE_DIR / "xfoil" / "xfoil.exe"

CSV_DIR.mkdir(parents=True, exist_ok=True)
POLAR_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Aerodynamic root-station selection
# -----------------------------

# True  -> run XFOIL for hub, mid, outer.
# False -> run XFOIL for inner, mid, outer.
USE_HUB_PROFILE_FOR_QPROP = True

ROOT_STATION = "hub" if USE_HUB_PROFILE_FOR_QPROP else "inner"
STATIONS = [ROOT_STATION, "mid", "outer"]


# -----------------------------
# Physical constants
# -----------------------------

RPM = 4000
OMEGA = RPM * 2.0 * np.pi / 60.0
NU_AIR = 1.5e-5  # air kinematic viscosity [m^2/s]

# Radial coordinates of the root construction and aerodynamic start stations.
INNER_STATION_RADIUS_MM = 4.5
HUB_OUTER_RADIUS_MM = 8.25

# Cubic spline setting used to evaluate the hub chord from inner/mid/outer chords.
# A natural spline is a stable choice for three radial control sections.
SPLINE_BC_TYPE = "natural"


# -----------------------------
# Reynolds-number discretisation
# -----------------------------

RE_MIN = 10_000
RE_MAX = 400_000
RE_ROUND = 5_000
RE_EXP = -0.5


# -----------------------------
# XFOIL run settings
# -----------------------------

NCRIT = 9
ALPHA_START = -5.0
ALPHA_END = 15.0
ALPHA_STEP = 0.5
XFOIL_ITER = 500
XFOIL_TIMEOUT_S = 60
MAX_WORKERS = min(os.cpu_count() or 4, 16)


# -----------------------------
# Polar validation and fitting settings
# -----------------------------

POLAR_MIN_BYTES = 400
FIT_ALPHA_LO = 0.0
FIT_ALPHA_HI = 8.0
CLA_MIN = 3.0
CLA_MAX = 8.5
CDMIN_MIN = 0.003
CDMIN_MAX = 0.08
STALL_BUFFER = 0.05
CL_JUMP_FILTER = 0.6
MIN_POLAR_POINTS = 5
MIN_FIT_POINTS = 4


# -----------------------------
# Output coefficient keys
# -----------------------------

AERO_KEYS = [
    "CL0", "CL_a", "CLmin", "CLmax", "CD0", "CD2u", "CD2l",
    "CLCD0", "REref", "REexp", "xfoil_ok",
]

print(f"Base directory : {BASE_DIR}")
print(f"Polar directory: {POLAR_DIR}")
print(f"XFOIL path     : {XFOIL_PATH}")
print(f"Workers        : {MAX_WORKERS}")
print(f"Root station   : {ROOT_STATION}")
print(f"Stations       : {STATIONS}")


Base directory : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset
Polar directory: c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\xfoil_polars
XFOIL path     : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\xfoil\xfoil.exe
Workers        : 16
Root station   : hub
Stations       : ['hub', 'mid', 'outer']


## 3. Load input data

Two input CSV files are required:

```text
./csv/prop_geometrical_params.csv
./csv/naca_codes.csv
```

The geometry file provides the radius, chord lengths, and middle radial position needed to compute Reynolds numbers.

The NACA file provides the airfoil codes for the available profile stations. The NACA columns are always read as strings so that leading zeros are preserved. For example, `0021` must remain `0021`, not `21`.


In [3]:
# Load geometry and NACA-code data

if not GEOMETRY_CSV_PATH.exists():
    raise FileNotFoundError(f"Geometry CSV not found: {GEOMETRY_CSV_PATH}")

if not NACA_CODES_CSV_PATH.exists():
    raise FileNotFoundError(f"NACA codes CSV not found: {NACA_CODES_CSV_PATH}")

geometry_df = pd.read_csv(GEOMETRY_CSV_PATH)

naca_dtype = {
    "naca_hub": "string",
    "naca_inner": "string",
    "naca_mid": "string",
    "naca_outer": "string",
}

naca_codes_df = pd.read_csv(NACA_CODES_CSV_PATH, dtype=naca_dtype)

available_naca_columns = [
    column for column in ["naca_hub", "naca_inner", "naca_mid", "naca_outer"]
    if column in naca_codes_df.columns
]

for column in available_naca_columns:
    naca_codes_df[column] = naca_codes_df[column].astype("string").str.zfill(4)

required_geometry_columns = [
    "config_id", "radius", "mid radial pos",
    "inner chord", "mid chord", "outer chord",
]

root_naca_column = f"naca_{ROOT_STATION}"
required_naca_columns = ["config_id", root_naca_column, "naca_mid", "naca_outer"]

missing_geometry_columns = [c for c in required_geometry_columns if c not in geometry_df.columns]
missing_naca_columns = [c for c in required_naca_columns if c not in naca_codes_df.columns]

if missing_geometry_columns:
    raise ValueError(f"Missing geometry columns: {missing_geometry_columns}")
if missing_naca_columns:
    raise ValueError(
        f"Missing NACA-code columns: {missing_naca_columns}. "
        f"Current root station is '{ROOT_STATION}'."
    )

input_df = pd.merge(
    geometry_df[required_geometry_columns],
    naca_codes_df[required_naca_columns],
    on="config_id",
    how="inner",
    validate="one_to_one",
)

input_df = input_df.sort_values("config_id").reset_index(drop=True)

print(f"Geometry rows : {len(geometry_df)}")
print(f"NACA rows     : {len(naca_codes_df)}")
print(f"Merged rows   : {len(input_df)}")
print(f"Root NACA col : {root_naca_column}")

input_df.head()


Geometry rows : 5000
NACA rows     : 5000
Merged rows   : 5000
Root NACA col : naca_hub


,config_id,radius,mid radial pos,inner chord,mid chord,outer chord,naca_hub,naca_mid,naca_outer
0,0,67,0.5,7,22,12,4617,4615,4712
1,1,60,0.6,11,17,16,6519,7516,8514
2,2,80,0.4,6,23,18,5518,4516,2411
3,3,66,0.5,5,12,10,7421,5318,2213
4,4,67,0.4,6,21,15,8418,7418,3517


## 4. Build station-level simulation table

XFOIL works on a single airfoil profile at a single Reynolds number. Therefore, each propeller configuration is expanded into three station-level requests.

If hub-root mode is active, the stations are:

$$
\{\text{hub}, \text{mid}, \text{outer}\}
$$

If inner-root mode is active, the stations are:

$$
\{\text{inner}, \text{mid}, \text{outer}\}
$$

The middle and outer radial positions are:

$$
r_\text{mid} = (\text{mid radial pos}) R
$$

$$
r_\text{outer} = R
$$

For the hub station, the NACA code comes from `naca_hub`. The chord is evaluated at the hub outer radius using the same cubic-spline idea used in the geometric pipeline:

$$
c_\text{hub} = S_c(r_\text{hub})
$$

where the spline passes through:

$$
(r_\text{inner}, c_\text{inner}), \quad
(r_\text{mid}, c_\text{mid}), \quad
(r_\text{outer}, c_\text{outer})
$$

This makes the Reynolds number at the hub station consistent with the aerodynamic blade start used later in QPROP.


In [4]:
# Helper functions for NACA parsing, cubic interpolation, and Reynolds-number calculation

def naca_to_components(naca_code: str) -> Tuple[int, int, int]:
    """Return camber, max-position, and thickness from a NACA 4-digit code."""
    code = str(naca_code).zfill(4)
    return int(code[0]), int(code[1]), int(code[2:4])


def cubic_spline_value(x_points: Iterable[float], y_points: Iterable[float], x_eval: float) -> float:
    """Evaluate a natural cubic spline at one radial position."""
    x = np.asarray(list(x_points), dtype=float)
    y = np.asarray(list(y_points), dtype=float)

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if len(np.unique(x)) != len(x):
        raise ValueError(f"Spline control radii must be unique. Received: {x.tolist()}")

    spline = CubicSpline(x, y, bc_type=SPLINE_BC_TYPE)
    return float(spline(float(x_eval)))


def station_reynolds_number(radius_mm: float, chord_mm: float) -> int:
    """Compute clipped and rounded Reynolds number for one blade station."""
    local_velocity = OMEGA * (radius_mm / 1000.0)
    reynolds = local_velocity * (chord_mm / 1000.0) / NU_AIR
    reynolds = np.clip(reynolds, RE_MIN, RE_MAX)
    return int(round(float(reynolds) / RE_ROUND) * RE_ROUND)


def build_station_simulation_table(df: pd.DataFrame) -> pd.DataFrame:
    """Expand each configuration into the selected XFOIL station requests."""
    records = []

    for _, row in df.iterrows():
        config_id = int(row["config_id"])
        radius_mm = float(row["radius"])
        mid_radius_mm = float(row["mid radial pos"]) * radius_mm

        inner_chord_mm = float(row["inner chord"])
        mid_chord_mm = float(row["mid chord"])
        outer_chord_mm = float(row["outer chord"])

        hub_chord_mm = cubic_spline_value(
            x_points=[INNER_STATION_RADIUS_MM, mid_radius_mm, radius_mm],
            y_points=[inner_chord_mm, mid_chord_mm, outer_chord_mm],
            x_eval=HUB_OUTER_RADIUS_MM,
        )

        if USE_HUB_PROFILE_FOR_QPROP:
            root_spec = (
                "hub",
                str(row["naca_hub"]).zfill(4),
                HUB_OUTER_RADIUS_MM,
                hub_chord_mm,
            )
        else:
            root_spec = (
                "inner",
                str(row["naca_inner"]).zfill(4),
                INNER_STATION_RADIUS_MM,
                inner_chord_mm,
            )

        station_specs = [
            root_spec,
            ("mid", str(row["naca_mid"]).zfill(4), mid_radius_mm, mid_chord_mm),
            ("outer", str(row["naca_outer"]).zfill(4), radius_mm, outer_chord_mm),
        ]

        for station, naca_code, radius_station_mm, chord_mm in station_specs:
            camber, max_pos, thickness = naca_to_components(naca_code)
            reynolds = station_reynolds_number(radius_station_mm, chord_mm)
            records.append({
                "config_id": config_id,
                "station": station,
                "naca_code": naca_code,
                "radius_station_mm": radius_station_mm,
                "chord_mm": chord_mm,
                "reynolds": reynolds,
                "camber": camber,
                "max_pos": max_pos,
                "thickness": thickness,
            })

    return pd.DataFrame(records)


station_df = build_station_simulation_table(input_df)

unique_pairs = (
    station_df[["naca_code", "reynolds"]]
    .drop_duplicates()
    .sort_values(["naca_code", "reynolds"])
    .itertuples(index=False, name=None)
)
unique_pairs = list(unique_pairs)

print(f"Station rows       : {len(station_df)}")
print(f"Unique XFOIL pairs : {len(unique_pairs)}")
print(f"Active stations    : {STATIONS}")

station_df.head()


Station rows       : 15000
Unique XFOIL pairs : 4787
Active stations    : ['hub', 'mid', 'outer']


,config_id,station,naca_code,radius_station_mm,chord_mm,reynolds,camber,max_pos,thickness
0,0,hub,4617,8.25,9.637490,10000,4,6,17
1,0,mid,4615,33.50,22.000000,20000,4,6,15
2,0,outer,4712,67.00,12.000000,20000,4,7,12
3,1,hub,6519,8.25,11.957828,10000,6,5,19
4,1,mid,7516,36.00,17.000000,15000,7,5,16


## 5. XFOIL execution and polar-file cache

Each polar file is named using the airfoil code, Reynolds number, and analysis type.

Examples:

```text
naca2412_re50000_visc_nc9.txt
naca2412_re50000_inv.txt
```

The cache avoids repeating expensive XFOIL runs. If a valid polar file already exists, the notebook reuses it automatically.

The viscous run is attempted first. If the viscous result cannot be parsed reliably, an inviscid run is attempted as a secondary source of lift-curve information.


In [5]:
# XFOIL cache and execution functions

def polar_path(naca_code: str, reynolds: int, viscous: bool) -> Path:
    """Return the expected polar-file path for one XFOIL run."""
    tag = f"visc_nc{NCRIT}" if viscous else "inv"
    return POLAR_DIR / f"naca{str(naca_code).zfill(4)}_re{int(reynolds)}_{tag}.txt"


def polar_valid(path: Path) -> bool:
    """Check whether a polar file exists and is large enough to contain data."""
    return path.exists() and path.stat().st_size > POLAR_MIN_BYTES


def build_xfoil_input(naca_code: str, polar_filename: str, reynolds: int, viscous: bool) -> str:
    """Build the command script passed to XFOIL through stdin."""
    viscous_block = f"VISC {int(reynolds)}\nVPAR\nN {NCRIT}\n\n" if viscous else ""
    return (
        "PLOP\nG F\n\n"
        f"NACA {str(naca_code).zfill(4)}\n"
        "OPER\n"
        f"ITER {XFOIL_ITER}\n"
        + viscous_block
        + f"PACC\n{polar_filename}\n\n"
        + f"ASEQ {ALPHA_START} {ALPHA_END} {ALPHA_STEP}\n"
        + "PACC\n\nQUIT\n"
    )


def run_one_polar(naca_code: str, reynolds: int, viscous: bool) -> None:
    """Run one XFOIL polar if it is not already cached."""
    output_path = polar_path(naca_code, reynolds, viscous)
    if polar_valid(output_path):
        return

    if not XFOIL_PATH.exists():
        raise FileNotFoundError(
            f"XFOIL executable not found at {XFOIL_PATH}. "
            "Install XFOIL there or update XFOIL_PATH in the constants cell."
        )

    if output_path.exists():
        output_path.unlink()

    xfoil_commands = build_xfoil_input(naca_code, output_path.name, reynolds, viscous)

    try:
        subprocess.run(
            [str(XFOIL_PATH)],
            input=xfoil_commands.encode(),
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=XFOIL_TIMEOUT_S,
            cwd=str(POLAR_DIR),
            check=False,
        )
    except subprocess.TimeoutExpired:
        pass


def run_parallel(pairs: Iterable[Tuple[str, int, bool]], description: str) -> None:
    """Run multiple XFOIL jobs in parallel."""
    pairs = list(pairs)
    if not pairs:
        print(f"{description}: nothing to run")
        return

    if not XFOIL_PATH.exists():
        raise FileNotFoundError(
            f"{description} requires {len(pairs)} new XFOIL runs, "
            f"but xfoil.exe was not found at {XFOIL_PATH}."
        )

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(run_one_polar, *pair): pair for pair in pairs}
        for future in tqdm(as_completed(futures), total=len(futures), desc=description):
            future.result()


## 6. Polar parsing and aerodynamic-coefficient fitting

The raw XFOIL polar is converted into a compact coefficient set.

The lift curve is fitted over a low-angle range where the airfoil is expected to behave approximately linearly:

$$
C_L(\alpha) \approx C_{L0} + C_{L\alpha}\alpha
$$

The drag polar is fitted as:

$$
C_D(C_L) \approx C_{D0} + C_{D2}(C_L - C_{L,CD0})^2
$$

The parser filters failed or unstable polar points and rejects physically unreasonable fits. This prevents a single bad XFOIL run from contaminating the downstream propeller model.


In [6]:
# Polar parsing and coefficient extraction

def _read_polar_rows(polar_file: Path, required_columns: int) -> np.ndarray:
    """Read numerical rows from an XFOIL polar file."""
    with open(polar_file, "r", encoding="utf-8", errors="ignore") as file:
        lines = file.readlines()

    data_start = next(
        (i + 1 for i, line in enumerate(lines) if line.strip().startswith("---") and i > 5),
        None,
    )
    if data_start is None:
        raise ValueError("No XFOIL data header found")

    rows = []
    for line in lines[data_start:]:
        line = line.strip()
        if not line:
            continue
        try:
            values = [float(value) for value in line.split()]
        except ValueError:
            continue
        if len(values) >= required_columns:
            rows.append(values[:required_columns])

    if len(rows) < MIN_POLAR_POINTS:
        raise ValueError(f"Only {len(rows)} valid polar rows found")
    return np.array(rows, dtype=float)


def _clean_lift_curve(alpha: np.ndarray, cl: np.ndarray, cd: np.ndarray | None = None):
    """Sort polar rows and remove large discontinuities in the lift curve."""
    order = np.argsort(alpha)
    alpha = alpha[order]
    cl = cl[order]
    if cd is not None:
        cd = cd[order]

    keep = np.concatenate([[True], np.abs(np.diff(cl)) < CL_JUMP_FILTER])
    alpha = alpha[keep]
    cl = cl[keep]
    if cd is not None:
        cd = cd[keep]

    if len(alpha) < MIN_POLAR_POINTS:
        raise ValueError("Too few polar points after lift-jump filtering")
    return (alpha, cl) if cd is None else (alpha, cl, cd)


def _linear_fit_mask(alpha: np.ndarray) -> np.ndarray:
    """Return the alpha range used for lift and drag fitting."""
    fit_mask = (alpha >= FIT_ALPHA_LO) & (alpha <= FIT_ALPHA_HI)
    if fit_mask.sum() < MIN_FIT_POINTS:
        fit_mask = (alpha >= -2.0) & (alpha <= FIT_ALPHA_HI)
    if fit_mask.sum() < MIN_FIT_POINTS:
        raise ValueError("Not enough points in the fitting alpha range")
    return fit_mask


def parse_viscous_polar(polar_file: Path, reynolds: int) -> Dict[str, float | int | bool]:
    """Parse a viscous XFOIL polar into aerodynamic coefficients."""
    try:
        rows = _read_polar_rows(polar_file, required_columns=3)
        alpha, cl, cd = rows[:, 0], rows[:, 1], rows[:, 2]
        positive_drag_mask = cd > 1e-6
        alpha, cl, cd = alpha[positive_drag_mask], cl[positive_drag_mask], cd[positive_drag_mask]
        if len(alpha) < MIN_POLAR_POINTS:
            raise ValueError("Too few polar points with positive drag")

        alpha, cl, cd = _clean_lift_curve(alpha, cl, cd)
        fit_mask = _linear_fit_mask(alpha)

        lift_fit = np.polyfit(alpha[fit_mask], cl[fit_mask], 1)
        cla_per_rad = lift_fit[0] * (180.0 / np.pi)
        cl0 = lift_fit[1]

        drag_fit = np.polyfit(cl[fit_mask], cd[fit_mask], 2)
        quadratic_term, linear_term, constant_term = drag_fit
        if quadratic_term > 1e-10:
            cd2 = float(quadratic_term)
            clcd0 = float(-linear_term / (2.0 * quadratic_term))
            cd0 = max(float(constant_term - linear_term**2 / (4.0 * quadratic_term)), 1e-5)
        else:
            cd0 = float(cd[fit_mask].min())
            clcd0 = float(cl[fit_mask][np.argmin(cd[fit_mask])])
            cd2 = 0.01

        if not (CLA_MIN <= cla_per_rad <= CLA_MAX):
            raise ValueError(f"Unphysical lift slope: CL_a = {cla_per_rad:.3f}")
        if not (CDMIN_MIN <= cd0 <= CDMIN_MAX):
            raise ValueError(f"Unphysical minimum drag: CD0 = {cd0:.5f}")

        return {
            "CL0": round(float(cl0), 4),
            "CL_a": round(float(cla_per_rad), 4),
            "CLmin": round(float(cl.min()) * (1.0 - STALL_BUFFER), 4),
            "CLmax": round(float(cl.max()) * (1.0 - STALL_BUFFER), 4),
            "CD0": round(float(cd0), 6),
            "CD2u": round(float(cd2), 6),
            "CD2l": round(float(cd2), 6),
            "CLCD0": round(float(clcd0), 4),
            "REref": int(reynolds),
            "REexp": RE_EXP,
            "xfoil_ok": True,
        }
    except Exception as error:
        return {"_err": str(error), "xfoil_ok": False}


def parse_inviscid_polar(polar_file: Path, reynolds: int, thickness: int) -> Dict[str, float | int | bool]:
    """Parse an inviscid polar and add empirical drag coefficients."""
    try:
        rows = _read_polar_rows(polar_file, required_columns=2)
        alpha, cl = rows[:, 0], rows[:, 1]
        alpha, cl = _clean_lift_curve(alpha, cl)
        fit_mask = _linear_fit_mask(alpha)

        lift_fit = np.polyfit(alpha[fit_mask], cl[fit_mask], 1)
        cla_per_rad = lift_fit[0] * (180.0 / np.pi)
        cl0 = lift_fit[1]
        if not (CLA_MIN <= cla_per_rad <= CLA_MAX):
            raise ValueError(f"Unphysical lift slope: CL_a = {cla_per_rad:.3f}")

        cd0_empirical = max(0.008, 0.005 + 0.0002 * thickness)
        cd2_empirical = 0.020
        return {
            "CL0": round(float(cl0), 4),
            "CL_a": round(float(cla_per_rad), 4),
            "CLmin": round(float(cl.min()) * (1.0 - STALL_BUFFER), 4),
            "CLmax": round(float(cl.max()) * (1.0 - STALL_BUFFER), 4),
            "CD0": round(float(cd0_empirical), 6),
            "CD2u": round(float(cd2_empirical), 6),
            "CD2l": round(float(cd2_empirical), 6),
            "CLCD0": round(float(cl0), 4),
            "REref": int(reynolds),
            "REexp": RE_EXP,
            "xfoil_ok": False,
        }
    except Exception as error:
        return {"_err": str(error), "xfoil_ok": False}


def thin_airfoil_fallback(camber: int, max_pos: int, thickness: int, reynolds: int) -> Dict[str, float | int | bool]:
    """Return a conservative analytical fallback if no XFOIL polar is usable."""
    camber_fraction = camber / 100.0
    cl0 = 2.0 * np.pi * 2.0 * camber_fraction
    cla_per_rad = 2.0 * np.pi
    cd0 = max(0.006, 0.004 + 0.0003 * thickness)
    return {
        "CL0": round(float(cl0), 4),
        "CL_a": round(float(cla_per_rad), 4),
        "CLmin": round(float(max(-0.6, -0.3 + cl0) * (1.0 - STALL_BUFFER)), 4),
        "CLmax": round(float(min(1.4, 0.9 + cl0) * (1.0 - STALL_BUFFER)), 4),
        "CD0": round(float(cd0), 6),
        "CD2u": 0.025,
        "CD2l": 0.025,
        "CLCD0": round(float(cl0), 4),
        "REref": int(reynolds),
        "REexp": RE_EXP,
        "xfoil_ok": False,
    }


def lookup_aero_coefficients(naca_code: str, reynolds: int, camber: int, max_pos: int, thickness: int) -> Dict[str, float | int | bool | str]:
    """Return the best available aerodynamic coefficients for one station."""
    viscous_path = polar_path(naca_code, reynolds, viscous=True)
    if polar_valid(viscous_path):
        aero = parse_viscous_polar(viscous_path, reynolds)
        if aero.get("xfoil_ok", False):
            aero["tier"] = "viscous"
            return aero

    inviscid_path = polar_path(naca_code, reynolds, viscous=False)
    if polar_valid(inviscid_path):
        aero = parse_inviscid_polar(inviscid_path, reynolds, thickness)
        if "_err" not in aero:
            aero["tier"] = "inviscid"
            return aero

    aero = thin_airfoil_fallback(camber, max_pos, thickness, reynolds)
    aero["tier"] = "thin_airfoil"
    return aero


## 7. Run XFOIL jobs

The batch execution has two phases:

1. Run all missing viscous polars.
2. For pairs where the viscous polar is unavailable or unusable, run the inviscid polar.

The actual number of XFOIL calls is usually much smaller than the number of station requests because many configurations share the same rounded Reynolds number and NACA code.


In [7]:
# Run missing XFOIL polars

t0 = time.time()

viscous_jobs = [
    (naca_code, reynolds, True)
    for naca_code, reynolds in unique_pairs
    if not polar_valid(polar_path(naca_code, reynolds, viscous=True))
]

cached_viscous = len(unique_pairs) - len(viscous_jobs)
print(f"Unique (NACA, Re) pairs : {len(unique_pairs)}")
print(f"Already cached viscous  : {cached_viscous}")
print(f"Viscous jobs to run     : {len(viscous_jobs)}")

if viscous_jobs:
    run_parallel(viscous_jobs, "XFOIL viscous")
else:
    print("XFOIL viscous: all required viscous polars are already cached")

inviscid_jobs = []
for naca_code, reynolds in unique_pairs:
    viscous_path = polar_path(naca_code, reynolds, viscous=True)
    viscous_ok = polar_valid(viscous_path) and parse_viscous_polar(viscous_path, reynolds).get("xfoil_ok", False)
    if not viscous_ok and not polar_valid(polar_path(naca_code, reynolds, viscous=False)):
        inviscid_jobs.append((naca_code, reynolds, False))

print(f"Inviscid jobs to run    : {len(inviscid_jobs)}")

if inviscid_jobs:
    run_parallel(inviscid_jobs, "XFOIL inviscid")
else:
    print("XFOIL inviscid: no additional inviscid polars required")

print(f"XFOIL stage completed in {time.time() - t0:.1f} s")


Unique (NACA, Re) pairs : 4787
Already cached viscous  : 4787
Viscous jobs to run     : 0
XFOIL viscous: all required viscous polars are already cached
Inviscid jobs to run    : 0
XFOIL inviscid: no additional inviscid polars required
XFOIL stage completed in 3.2 s


## 8. Assemble configuration-level aerodynamic table

After the polar files are available, each selected station is assigned the best available aerodynamic coefficient set.

The output table has one row per propeller configuration.

For each selected station, the table stores:

- NACA code,
- Reynolds number,
- robustness tier,
- fitted lift and drag coefficients.

The station names depend on the root-selection constant:

- `hub`, `mid`, `outer` when hub-root mode is active;
- `inner`, `mid`, `outer` when inner-root mode is active.

This makes the output directly compatible with the next pipeline stage that generates QPROP input files.


In [8]:
# Assemble one output row per propeller configuration

records = []
station_records = station_df.to_dict("records")
station_records.sort(key=itemgetter("config_id"))

for config_id, group in tqdm(
    groupby(station_records, key=itemgetter("config_id")),
    total=input_df["config_id"].nunique(),
    desc="Assembling XFOIL output",
):
    row_output = {"config_id": int(config_id)}

    for station_row in group:
        station = station_row["station"]
        naca_code = str(station_row["naca_code"]).zfill(4)
        reynolds = int(station_row["reynolds"])
        camber = int(station_row["camber"])
        max_pos = int(station_row["max_pos"])
        thickness = int(station_row["thickness"])

        aero = lookup_aero_coefficients(naca_code, reynolds, camber, max_pos, thickness)

        row_output[f"naca_{station}"] = naca_code
        row_output[f"re_{station}"] = reynolds
        row_output[f"tier_{station}"] = aero.get("tier", "unknown")

        for key in AERO_KEYS:
            row_output[f"{station}_{key}"] = aero[key]

    records.append(row_output)

xfoil_results_df = pd.DataFrame(records)

ordered_columns = ["config_id"]
for station in STATIONS:
    ordered_columns.extend([f"naca_{station}", f"re_{station}", f"tier_{station}"])
for station in STATIONS:
    ordered_columns.extend([f"{station}_{key}" for key in AERO_KEYS])

xfoil_results_df = xfoil_results_df[ordered_columns]
xfoil_results_df = xfoil_results_df.sort_values("config_id").reset_index(drop=True)

for station in STATIONS:
    xfoil_results_df[f"naca_{station}"] = xfoil_results_df[f"naca_{station}"].astype("string").str.zfill(4)

xfoil_results_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved XFOIL results to: {OUTPUT_CSV_PATH}")
print(f"Rows                 : {len(xfoil_results_df)}")
print(f"Columns              : {len(xfoil_results_df.columns)}")
print(f"Root station         : {ROOT_STATION}")

xfoil_results_df.head()


Assembling XFOIL output: 100%|██████████| 5000/5000 [00:07<00:00, 671.62it/s]


Saved XFOIL results to: c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\xfoil_profile_simulation.csv
Rows                 : 5000
Columns              : 43
Root station         : hub


,config_id,naca_hub,re_hub,tier_hub,naca_mid,re_mid,tier_mid,naca_outer,re_outer,tier_outer,...,outer_CL_a,outer_CLmin,outer_CLmax,outer_CD0,outer_CD2u,outer_CD2l,outer_CLCD0,outer_REref,outer_REexp,outer_xfoil_ok
0,0,4617,10000,viscous,4615,20000,viscous,4712,20000,viscous,...,4.0000,-0.4253,0.8684,0.035460,0.251010,0.251010,0.1060,20000,-0.5,True
1,1,6519,10000,inviscid,7516,15000,viscous,8514,25000,viscous,...,6.1848,-0.3958,0.9688,0.075877,0.102049,0.102049,-0.0473,25000,-0.5,True
2,2,5518,10000,viscous,4516,20000,viscous,2411,40000,viscous,...,8.1089,-0.4415,1.0536,0.023710,0.010000,0.010000,-0.0249,40000,-0.5,True
3,3,7421,10000,inviscid,5318,10000,viscous,2213,20000,inviscid,...,6.9417,-0.3646,1.9225,0.008000,0.020000,0.020000,0.2262,20000,-0.5,False
4,4,8418,10000,inviscid,7418,15000,inviscid,3517,30000,viscous,...,4.2840,-0.5403,0.6678,0.040446,0.124572,0.124572,-0.3460,30000,-0.5,True


## 9. Final checks

The final checks verify that:

- every input `config_id` appears in the output;
- the output is sorted and one-to-one with the input configurations;
- the selected station set matches the selected root mode;
- NACA codes are valid four-character strings;
- Reynolds numbers stay inside the selected XFOIL range;
- all required polar folders and CSV outputs exist;
- every station has a clearly assigned robustness tier.


In [9]:
# Final consistency checks

all_ok = True


def check(condition: bool, message: str) -> None:
    """Print a compact pass/fail message."""
    global all_ok
    status = "OK" if condition else "FAIL"
    print(f"[{status:4s}] {message}")
    if not condition:
        all_ok = False


input_ids = input_df["config_id"].astype(int).tolist()
output_ids = xfoil_results_df["config_id"].astype(int).tolist()

check(len(xfoil_results_df) == len(input_df), "output row count matches input row count")
check(output_ids == input_ids, "config_id order and alignment are preserved")
check(xfoil_results_df["config_id"].is_unique, "config_id is unique in output")

expected_station_set = {"hub", "mid", "outer"} if USE_HUB_PROFILE_FOR_QPROP else {"inner", "mid", "outer"}
actual_station_set = set(STATIONS)
check(actual_station_set == expected_station_set, "active station set matches the selected root mode")


def is_valid_naca_string(value: object) -> bool:
    text = str(value)
    return len(text) == 4 and text.isdigit()


for station in STATIONS:
    naca_column = f"naca_{station}"
    check(
        xfoil_results_df[naca_column].map(is_valid_naca_string).all(),
        f"{naca_column} contains valid four-digit NACA strings",
    )

for station in STATIONS:
    re_column = f"re_{station}"
    check(
        xfoil_results_df[re_column].between(RE_MIN, RE_MAX).all(),
        f"{re_column} lies within [{RE_MIN}, {RE_MAX}]",
    )

valid_tiers = {"viscous", "inviscid", "thin_airfoil"}
for station in STATIONS:
    tier_column = f"tier_{station}"
    check(
        set(xfoil_results_df[tier_column].dropna()).issubset(valid_tiers),
        f"{tier_column} contains only known robustness tiers",
    )

check(POLAR_DIR.exists(), "polar directory exists")
check(OUTPUT_CSV_PATH.exists(), "output CSV exists")

print()
print("Selected aerodynamic station set")
print("-" * 72)
print(f"USE_HUB_PROFILE_FOR_QPROP = {USE_HUB_PROFILE_FOR_QPROP}")
print(f"ROOT_STATION              = {ROOT_STATION}")
print(f"STATIONS                  = {STATIONS}")

print()
print("Tier summary")
print("-" * 72)
for station in STATIONS:
    tier_column = f"tier_{station}"
    counts = xfoil_results_df[tier_column].value_counts().reindex(sorted(valid_tiers), fill_value=0)
    print(f"{station:>6s}: " + ", ".join(f"{tier}={count}" for tier, count in counts.items()))

print()
print("=" * 72)
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED")
print("=" * 72)


[OK  ] output row count matches input row count
[OK  ] config_id order and alignment are preserved
[OK  ] config_id is unique in output
[OK  ] active station set matches the selected root mode
[OK  ] naca_hub contains valid four-digit NACA strings
[OK  ] naca_mid contains valid four-digit NACA strings
[OK  ] naca_outer contains valid four-digit NACA strings
[OK  ] re_hub lies within [10000, 400000]
[OK  ] re_mid lies within [10000, 400000]
[OK  ] re_outer lies within [10000, 400000]
[OK  ] tier_hub contains only known robustness tiers
[OK  ] tier_mid contains only known robustness tiers
[OK  ] tier_outer contains only known robustness tiers
[OK  ] polar directory exists
[OK  ] output CSV exists

Selected aerodynamic station set
------------------------------------------------------------------------
USE_HUB_PROFILE_FOR_QPROP = True
ROOT_STATION              = hub
STATIONS                  = ['hub', 'mid', 'outer']

Tier summary
----------------------------------------------------------